In [2]:
# ============================================================
# CELL 1 — SETUP + LOAD DATA
# ============================================================
%pip install -q -U unsloth sentence-transformers faiss-cpu rank-bm25 \
    accelerate peft bitsandbytes trl datasets huggingface_hub \
    sacrebleu rapidfuzz bert-score==0.3.13

from google.colab import drive
drive.mount("/content/drive")

import os, glob, json, re, gc, pickle, unicodedata
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
MODEL_FOLDER = "Mistral_7B_Instruct_v0.3"

ROOT = Path("/content/drive/MyDrive/Bangla_Agriculture_FT_RAG_Mistral")
OUT = ROOT / MODEL_FOLDER
MODEL_DIR = OUT / "model_adapter"
INDEX_DIR = OUT / "retrieval_index"
CHECKPOINT_DIR = OUT / "checkpoints"

for p in [OUT, MODEL_DIR, INDEX_DIR, CHECKPOINT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TRAIN_NAME = "Bangla_Agriculture_QA_Train_800.json"
TEST_NAME  = "Bangla_Agriculture_QA_Test_200.json"
KB_NAME    = "Bangla_Agriculture_Knowledge_Base_500.json"

def locate(name):
    candidates = [
        Path("/content") / name,
        Path("/content/drive/MyDrive") / name,
        Path.cwd() / name,
    ]
    for p in candidates:
        if p.exists():
            return p
    matches = list(Path("/content").rglob(name)) if Path("/content").exists() else []
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Upload or copy {name} to /content or Google Drive.")

TRAIN_PATH = locate(TRAIN_NAME)
TEST_PATH  = locate(TEST_NAME)
KB_PATH    = locate(KB_NAME)

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

train_obj = load_json(TRAIN_PATH)
test_obj = load_json(TEST_PATH)
knowledge_base = load_json(KB_PATH)

train_raw = train_obj["qa_pairs"] if isinstance(train_obj, dict) and "qa_pairs" in train_obj else train_obj
test_raw  = test_obj["qa_pairs"] if isinstance(test_obj, dict) and "qa_pairs" in test_obj else test_obj

train_rows = []
for row in train_raw:
    q = str(row.get("question", "")).strip()
    a = str(row.get("reference_answer", "")).strip()
    if q and a:
        train_rows.append({
            "id": str(row.get("id", "")),
            "question": q,
            "answer": a,
        })

tests = pd.DataFrame([
    {
        "id": str(r.get("id", "")),
        "domain": "agriculture",
        "question": str(r.get("question", "")).strip(),
        "gold": str(r.get("reference_answer", "")).strip(),
    }
    for r in test_raw
])

if len(train_rows) == 0:
    raise ValueError("No valid training QA pairs found.")
if len(tests) == 0 or tests["gold"].str.strip().eq("").any():
    raise ValueError("Test data is empty or contains blank reference answers.")
if not isinstance(knowledge_base, list) or len(knowledge_base) == 0:
    raise ValueError("Knowledge base must be a non-empty JSON list.")

with open(OUT / "unified_train_used.json", "w", encoding="utf-8") as f:
    json.dump(train_rows, f, ensure_ascii=False, indent=2)

tests.to_csv(OUT / "test_questions.csv", index=False, encoding="utf-8-sig")

print("Model:", MODEL_NAME)
print("Training QA:", len(train_rows))
print("Test QA:", len(tests))
print("Knowledge-base records:", len(knowledge_base))
print("Output:", OUT)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# ============================================================
# CELL 2 — QLORA FINE-TUNING
# Preserves the supplied Mistral FT configuration.
# Reuses the saved adapter unless FORCE_RETRAIN=True.
# ============================================================
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from huggingface_hub import notebook_login

# Mistral access may require you to be logged in / have accepted its HF terms.
# If already authenticated in Colab, this can be skipped.
try:
    notebook_login()
except Exception as e:
    print("HF login skipped/failed:", e)

FORCE_RETRAIN = False
MAX_SEQ_LENGTH = 1024

SYSTEM_PROMPT_FT = (
    "তুমি বাংলাদেশের কৃষি বিষয়ে একজন সহায়ক সহকারী। "
    "শুধুমাত্র নির্ভরযোগ্য তথ্য দাও। "
    "যদি তথ্য জানা না থাকে, বলবে 'আমি জানি না।'"
)

adapter_ready = (MODEL_DIR / "adapter_config.json").exists()

if adapter_ready and not FORCE_RETRAIN:
    print("Existing fine-tuned adapter found. Skipping training:")
    print(MODEL_DIR)

else:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407
    )

    def formatting(example):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT_FT},
            {"role": "user", "content": example["question"]},
            {"role": "assistant", "content": example["answer"]},
        ]
        return {
            "text": tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
        }

    dataset = Dataset.from_list(train_rows).map(
        formatting,
        remove_columns=["id", "question", "answer"]
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=True,
        args=SFTConfig(
            output_dir=str(CHECKPOINT_DIR),
            num_train_epochs=5,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            learning_rate=2e-4,
            optim="adamw_8bit",
            lr_scheduler_type="cosine",
            warmup_ratio=0.03,
            weight_decay=0.01,
            fp16=False,
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10,
            save_strategy="epoch",
            save_total_limit=3,
            report_to="none",
            seed=3407,
        ),
    )

    checkpoints = sorted(
        CHECKPOINT_DIR.glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1
    )
    resume = str(checkpoints[-1]) if checkpoints else None
    print("Resume checkpoint:", resume)

    train_result = trainer.train(resume_from_checkpoint=resume)

    model.save_pretrained(str(MODEL_DIR))
    tokenizer.save_pretrained(str(MODEL_DIR))

    pd.DataFrame(trainer.state.log_history).to_csv(
        OUT / "training_log.csv",
        index=False
    )

    config = {
        "base_model": MODEL_NAME,
        "method": "QLoRA with Unsloth",
        "epochs": 5,
        "max_seq_length": MAX_SEQ_LENGTH,
        "lora_r": 16,
        "lora_alpha": 16,
        "learning_rate": 2e-4,
        "batch_size": 2,
        "gradient_accumulation_steps": 4,
        "packing": True,
        "seed": 3407,
    }

    with open(OUT / "training_config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)

    print("Fine-tuning complete. Adapter saved to:", MODEL_DIR)

    del trainer, model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.11: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth 2026.9.11 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  return original(name)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.528363
20,0.914645
30,0.718381
40,0.543709
50,0.455289
60,0.364171
70,0.322146
80,0.286991


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Bangla_Agriculture_FT_RAG_Mistral/Mistral_7B_Instruct_v0.3/checkpoints/checkpoint-16/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/Bangla_Agriculture_FT_RAG_Mistral/Mistral_7B_Instruct_v0.3/checkpoints/checkpoint-16.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Bangla_Agriculture_FT_RAG_Mistral/Mistral_7B_Instruct_v0.3/checkpoints/checkpoint-32/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/Bangla_Agriculture_FT_RAG_Mistral/Mistral_7B_Instruct_v0.3/checkpoints/checkpoint-32.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Bangla_Agriculture_FT_RAG_Mistral/Mistral_7B_Instruct_v0.3/checkpoints/checkpoint-48/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/Bangla_Agriculture_FT_RAG_Mistral

Fine-tuning complete. Adapter saved to: /content/drive/MyDrive/Bangla_Agriculture_FT_RAG_Mistral/Mistral_7B_Instruct_v0.3/model_adapter


In [4]:
# ============================================================
# CELL 3 — CHUNK KB + BUILD / REUSE RETRIEVAL INDEX
# BGE-M3 dense embeddings + FAISS + BM25
# ============================================================
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import faiss

CHUNKS_FILE = INDEX_DIR / "chunks.json"
FAISS_FILE = INDEX_DIR / "bge_m3.faiss"
EMB_FILE = INDEX_DIR / "bge_m3_embeddings.npy"
BM25_FILE = INDEX_DIR / "bm25.pkl"

def chunk_text(text, size=180, overlap=30):
    words = str(text).split()
    if len(words) <= size:
        return [str(text).strip()]

    pieces = []
    start = 0
    while start < len(words):
        end = min(start + size, len(words))
        pieces.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap
    return pieces

chunks = []

for row in knowledge_base:
    text = str(row.get("content", "")).strip()
    if not text:
        continue

    doc_id = str(row.get("id", f"kb_{len(chunks)}"))
    title = str(row.get("title", "")).strip()

    for text_chunk in chunk_text(text):
        chunks.append({
            "chunk_id": len(chunks),
            "doc_id": doc_id,
            "domain": "agriculture",
            "title": title,
            "source_site": str(row.get("source_site", "")),
            "source_title": str(row.get("source_title", "")),
            "source_url": str(row.get("source_url", "")),
            "text": text_chunk,
            "index_text": f"শিরোনাম: {title}\nতথ্য: {text_chunk}",
        })

if not chunks:
    raise ValueError("No chunks were created from the knowledge base.")

with open(CHUNKS_FILE, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

BN_TO_EN = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

def tokenize_for_retrieval(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.translate(BN_TO_EN).lower()
    return re.findall(r"[\u0980-\u09FF]+|[a-z]+|\d+(?:\.\d+)?", text)

REBUILD_INDEX = False

if FAISS_FILE.exists() and BM25_FILE.exists() and not REBUILD_INDEX:
    faiss_index = faiss.read_index(str(FAISS_FILE))
    with open(BM25_FILE, "rb") as f:
        bm25 = pickle.load(f)["bm25"]

    if faiss_index.ntotal != len(chunks) or bm25.corpus_size != len(chunks):
        print("Stored index size does not match current chunks. Rebuilding.")
        REBUILD_INDEX = True
    else:
        print("Reusing existing retrieval indices.")

if REBUILD_INDEX or not (FAISS_FILE.exists() and BM25_FILE.exists()):
    texts = [x["index_text"] for x in chunks]
    device = "cuda" if torch.cuda.is_available() else "cpu"

    embedder = SentenceTransformer("BAAI/bge-m3", device=device)

    embeddings = embedder.encode(
        texts,
        batch_size=8,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
    faiss_index.add(embeddings)

    faiss.write_index(faiss_index, str(FAISS_FILE))
    np.save(EMB_FILE, embeddings)

    tokenized_corpus = [tokenize_for_retrieval(x) for x in texts]
    bm25 = BM25Okapi(tokenized_corpus)

    with open(BM25_FILE, "wb") as f:
        pickle.dump(
            {"bm25": bm25, "tokens": tokenized_corpus},
            f
        )

    del embedder, embeddings
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Chunks:", len(chunks))
print("FAISS vectors:", faiss_index.ntotal)
print("BM25 documents:", bm25.corpus_size)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Chunks: 500
FAISS vectors: 500
BM25 documents: 500


In [5]:
# ============================================================
# CELL 4 — HYBRID RETRIEVAL + RRF + BGE RERANKING
# Final top-3 passages per test question
# ============================================================
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

with open(CHUNKS_FILE, encoding="utf-8") as f:
    chunks = json.load(f)

faiss_index = faiss.read_index(str(FAISS_FILE))

with open(BM25_FILE, "rb") as f:
    bm25 = pickle.load(f)["bm25"]

device = "cuda" if torch.cuda.is_available() else "cpu"

embedder = SentenceTransformer("BAAI/bge-m3", device=device)

RERANKER_NAME = "BAAI/bge-reranker-v2-m3"

rerank_tokenizer = AutoTokenizer.from_pretrained(RERANKER_NAME)

reranker = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)

reranker.eval()

def hybrid_retrieve(question, domain="agriculture", candidate_k=25):
    qvec = embedder.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    dense_scores, dense_ids = faiss_index.search(
        qvec,
        faiss_index.ntotal
    )

    dense_rank = []

    for score, idx in zip(dense_scores[0], dense_ids[0]):
        idx = int(idx)
        if chunks[idx]["domain"] == domain:
            dense_rank.append((idx, float(score)))
        if len(dense_rank) >= 100:
            break

    bm25_scores = bm25.get_scores(
        tokenize_for_retrieval(question)
    )

    sparse_ids = np.argsort(bm25_scores)[::-1]
    sparse_rank = []

    for idx in sparse_ids:
        idx = int(idx)
        if chunks[idx]["domain"] == domain:
            sparse_rank.append((idx, float(bm25_scores[idx])))
        if len(sparse_rank) >= 100:
            break

    rrf = defaultdict(float)
    dense_map = {}
    bm25_map = {}

    for rank, (idx, score) in enumerate(dense_rank, 1):
        rrf[idx] += 1 / (60 + rank)
        dense_map[idx] = score

    for rank, (idx, score) in enumerate(sparse_rank, 1):
        rrf[idx] += 1 / (60 + rank)
        bm25_map[idx] = score

    best_ids = sorted(
        rrf,
        key=rrf.get,
        reverse=True
    )[:candidate_k]

    candidates = []

    for idx in best_ids:
        item = chunks[idx].copy()
        item["dense_score"] = dense_map.get(idx, 0.0)
        item["bm25_score"] = bm25_map.get(idx, 0.0)
        item["rrf_score"] = rrf[idx]
        candidates.append(item)

    return candidates

def rerank(question, candidates, top_k=3):
    scores = []

    for start in range(0, len(candidates), 4):
        batch = candidates[start:start + 4]

        encoded = rerank_tokenizer(
            [question] * len(batch),
            [x["index_text"] for x in batch],
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(device)

        with torch.inference_mode():
            logits = reranker(**encoded).logits.squeeze(-1)

        scores.extend(
            logits.float().cpu().tolist()
        )

    for item, score in zip(candidates, scores):
        item["reranker_score"] = float(score)

    return sorted(
        candidates,
        key=lambda x: x["reranker_score"],
        reverse=True,
    )[:top_k]

RETRIEVAL_JSON = OUT / "retrievals.json"
RETRIEVAL_CSV = OUT / "retrievals.csv"
REUSE_RETRIEVALS = True

if RETRIEVAL_JSON.exists() and REUSE_RETRIEVALS:
    with open(RETRIEVAL_JSON, encoding="utf-8") as f:
        retrieval_cache = json.load(f)

    if len(retrieval_cache) != len(tests):
        print("Saved retrieval count mismatch. Rebuilding.")
        retrieval_cache = []
else:
    retrieval_cache = []

if not retrieval_cache:
    retrieval_cache = []

    for _, row in tqdm(
        tests.iterrows(),
        total=len(tests),
        desc="Hybrid retrieval + reranking",
    ):
        candidates = hybrid_retrieve(
            row["question"],
            row["domain"]
        )

        docs = rerank(
            row["question"],
            candidates,
            top_k=3
        )

        retrieval_cache.append(docs)

    with open(RETRIEVAL_JSON, "w", encoding="utf-8") as f:
        json.dump(
            retrieval_cache,
            f,
            ensure_ascii=False,
            indent=2
        )

retrieval_rows = []

for row_idx, row in tests.iterrows():
    docs = retrieval_cache[row_idx]

    x = {
        "id": row["id"],
        "domain": row["domain"],
        "question": row["question"],
    }

    for i, doc in enumerate(docs, 1):
        x[f"doc_{i}"] = doc["doc_id"]
        x[f"title_{i}"] = doc["title"]
        x[f"context_{i}"] = doc["text"]
        x[f"source_{i}"] = doc["source_url"]
        x[f"dense_score_{i}"] = doc["dense_score"]
        x[f"bm25_score_{i}"] = doc["bm25_score"]
        x[f"rrf_score_{i}"] = doc["rrf_score"]
        x[f"reranker_score_{i}"] = doc["reranker_score"]

    retrieval_rows.append(x)

pd.DataFrame(retrieval_rows).to_csv(
    RETRIEVAL_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("Retrievals ready:", len(retrieval_cache))
print("Saved:", RETRIEVAL_CSV)

del embedder, reranker, rerank_tokenizer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Hybrid retrieval + reranking:   0%|          | 0/200 [00:00<?, ?it/s]

Retrievals ready: 200
Saved: /content/drive/MyDrive/Bangla_Agriculture_FT_RAG_Mistral/Mistral_7B_Instruct_v0.3/retrievals.csv


In [6]:
# ============================================================
# CELL 5 — FINE-TUNED MISTRAL + RAG GENERATION
# Uses the saved QLoRA adapter and retrieved top-3 contexts.
# ============================================================
from unsloth import FastLanguageModel

if not (MODEL_DIR / "adapter_config.json").exists():
    raise FileNotFoundError(
        f"Fine-tuned adapter not found at {MODEL_DIR}. Run Cell 2 first."
    )

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(MODEL_DIR),
    max_seq_length=8192,
    dtype=None,
    load_in_4bit=True
)

FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

MAX_NEW_TOKENS = 320

SYSTEM_PROMPT_RAG = (
    "তুমি বাংলাদেশের কৃষি বিষয়ক প্রশ্নোত্তর সহকারী। "
    "তুমি এই কাজের জন্য fine-tuned, কিন্তু চূড়ান্ত উত্তরে শুধুমাত্র প্রদত্ত Context ব্যবহার করবে। "
    "Context-এর মধ্যে প্রশ্নের সঠিক উত্তর যে বাক্য বা তথ্যের মধ্যে আছে, "
    "সেটি যথাসম্ভব হুবহু ব্যবহার করে উত্তর দাও। "
    "নিজের ভাষায় অপ্রয়োজনীয়ভাবে পুনর্লিখন বা ব্যাখ্যা করো না। "
    "প্রশ্নে যতটুকু তথ্য চাওয়া হয়েছে শুধু ততটুকুই দাও। "
    "সংখ্যা, পরিমাণ, একক, সময়, জাতের নাম এবং অন্যান্য তথ্য "
    "Context-এ যেভাবে আছে সেভাবেই রাখো। "
    "একাধিক Context থাকলে প্রশ্নের সাথে সবচেয়ে সরাসরি সম্পর্কিত "
    "Context-এর তথ্যকে অগ্রাধিকার দাও। "
    "Context-এ উত্তর একেবারেই না থাকলে শুধু বলো: "
    "'প্রদত্ত তথ্যে এই প্রশ্নের উত্তর পাওয়া যায়নি।'"
)

stop_token_ids = model.generation_config.eos_token_id
if stop_token_ids is None:
    stop_token_ids = tokenizer.eos_token_id
if isinstance(stop_token_ids, int):
    stop_token_ids = [stop_token_ids]

def generate_ft_rag_answer(question, docs):
    context = "\n\n".join([
        f"[Context {i}]\nশিরোনাম: {doc['title']}\nতথ্য: {doc['text']}"
        for i, doc in enumerate(docs, 1)
    ])

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT_RAG
        },
        {
            "role": "user",
            "content":
                f"Context:\n{context}\n\n"
                f"প্রশ্ন: {question}\n\n"
                "উত্তর:"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors="pt",
        truncation=True,
        max_length=7000,
    ).to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=stop_token_ids,
        )

    generated = output[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    truncated = (
        len(generated) >= MAX_NEW_TOKENS
        and generated[-1].item() not in stop_token_ids
    )

    return answer, truncated

PARTIAL = OUT / "predictions_partial.csv"
FINAL_PRED = OUT / "predictions.csv"

done = {}

if PARTIAL.exists():
    for _, r in pd.read_csv(PARTIAL).fillna("").iterrows():
        done[str(r["id"])] = r.to_dict()

    print(
        "Resuming from",
        len(done),
        "saved predictions."
    )

predictions = []

for i, row in tqdm(
    tests.iterrows(),
    total=len(tests),
    desc="Fine-tuned Mistral + RAG",
):
    key = str(row["id"])
    result = done.get(key)

    if result is None:
        answer, truncated = generate_ft_rag_answer(
            row["question"],
            retrieval_cache[i]
        )

        result = {
            "id": key,
            "domain": row["domain"],
            "question": row["question"],
            "gold": row["gold"],
            "prediction": answer,
            "truncated": truncated,
        }

    predictions.append(result)

    pd.DataFrame(predictions).to_csv(
        PARTIAL,
        index=False,
        encoding="utf-8-sig",
    )

pred_df = pd.DataFrame(predictions)

pred_df.to_csv(
    FINAL_PRED,
    index=False,
    encoding="utf-8-sig",
)

print("Completed:", len(pred_df))
print(
    "Truncated:",
    pred_df["truncated"]
    .astype(str)
    .str.lower()
    .eq("true")
    .sum()
)
print("Saved:", FINAL_PRED)

del model, tokenizer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


==((====))==  Unsloth 2026.9.11: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Fine-tuned Mistral + RAG:   0%|          | 0/200 [00:00<?, ?it/s]

Both `max_new_tokens` (=320) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Completed: 200
Truncated: 0
Saved: /content/drive/MyDrive/Bangla_Agriculture_FT_RAG_Mistral/Mistral_7B_Instruct_v0.3/predictions.csv


In [7]:
# ============================================================
# CELL 6 — FINAL EVALUATION
# Same metrics as the supplied Mistral FT and RAG notebooks.
# ============================================================
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score

df = pd.read_csv(
    OUT / "predictions.csv"
).fillna("")

def normalize(text):
    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    text = text.translate(
        BN_TO_EN
    ).lower()

    text = re.sub(
        r"[^\u0980-\u09FFA-Za-z0-9]+",
        " ",
        text
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()

def tokens(text):
    return normalize(text).split()

def exact_match(pred, gold):
    return float(
        normalize(pred) == normalize(gold)
    )

def token_f1(pred, gold):
    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    overlap = sum(
        (Counter(p) & Counter(g)).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(g)

    return (
        2 * precision * recall
        / (precision + recall)
    )

def rouge_n(pred, gold, n):
    p = tokens(pred)
    g = tokens(gold)

    if len(p) < n or len(g) < n:
        return 0.0

    pn = Counter(
        tuple(p[i:i+n])
        for i in range(len(p)-n+1)
    )

    gn = Counter(
        tuple(g[i:i+n])
        for i in range(len(g)-n+1)
    )

    overlap = sum(
        (pn & gn).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / sum(pn.values())
    recall = overlap / sum(gn.values())

    return (
        2 * precision * recall
        / (precision + recall)
    )

def rouge_l(pred, gold):
    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    dp = [0] * (len(g) + 1)

    for x in p:
        new = [0]

        for j, y in enumerate(g, 1):
            if x == y:
                new.append(
                    dp[j-1] + 1
                )
            else:
                new.append(
                    max(
                        dp[j],
                        new[-1]
                    )
                )

        dp = new

    lcs = dp[-1]

    precision = lcs / len(p)
    recall = lcs / len(g)

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        / (precision + recall)
    )

df["Exact Match"] = [
    exact_match(p, g)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["Fuzzy Match"] = [
    fuzz.token_set_ratio(
        normalize(p),
        normalize(g)
    ) / 100
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["Token F1"] = [
    token_f1(p, g)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["ROUGE-1"] = [
    rouge_n(p, g, 1)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["ROUGE-2"] = [
    rouge_n(p, g, 2)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["ROUGE-L"] = [
    rouge_l(p, g)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

bleu = BLEU(
    tokenize="none",
    smooth_method="exp",
    effective_order=True
)

pred_texts = [
    " ".join(tokens(x))
    for x in df["prediction"]
]

gold_texts = [
    " ".join(tokens(x))
    for x in df["gold"]
]

corpus_bleu = (
    bleu.corpus_score(
        pred_texts,
        [gold_texts]
    ).score
    / 100
)

print("Calculating multilingual BERTScore...")

# CPU is retained to match the supplied Mistral evaluation notebooks.
# Change to device="cuda" only if you intentionally want faster evaluation
# and have enough free VRAM at this stage.
P, R, F1 = bert_score(
    df["prediction"].astype(str).tolist(),
    df["gold"].astype(str).tolist(),
    model_type="bert-base-multilingual-cased",
    batch_size=4,
    device="cpu",
    idf=False,
    rescale_with_baseline=False,
    verbose=True,
)

df["BERT Precision"] = P.cpu().numpy()
df["BERT Recall"] = R.cpu().numpy()
df["BERT F1"] = F1.cpu().numpy()

result = pd.DataFrame({
    "metric": [
        "Exact Match",
        "Fuzzy Match",
        "Corpus BLEU",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "Token F1",
        "BERT Precision",
        "BERT Recall",
        "BERT F1",
        "Truncated Outputs",
    ],
    "score": [
        df["Exact Match"].mean(),
        df["Fuzzy Match"].mean(),
        corpus_bleu,
        df["ROUGE-1"].mean(),
        df["ROUGE-2"].mean(),
        df["ROUGE-L"].mean(),
        df["Token F1"].mean(),
        df["BERT Precision"].mean(),
        df["BERT Recall"].mean(),
        df["BERT F1"].mean(),
        df["truncated"]
        .astype(str)
        .str.lower()
        .eq("true")
        .sum(),
    ],
})

df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

result.to_csv(
    OUT / "result.csv",
    index=False,
    encoding="utf-8-sig",
)

display(result)

print("\nSaved:")
print(OUT / "retrievals.csv")
print(OUT / "retrievals.json")
print(OUT / "predictions.csv")
print(OUT / "result.csv")


Calculating multilingual BERTScore...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

computing greedy matching.


  0%|          | 0/50 [00:00<?, ?it/s]

done in 11.03 seconds, 18.14 sentences/sec


,metric,score
0,Exact Match,0.615000
1,Fuzzy Match,0.871110
2,Corpus BLEU,0.672739
3,ROUGE-1,0.750409
4,ROUGE-2,0.723824
5,ROUGE-L,0.743542
6,Token F1,0.750409
7,BERT Precision,0.919383
8,BERT Recall,0.902252
9,BERT F1,0.910037



Saved:
/content/drive/MyDrive/Bangla_Agriculture_FT_RAG_Mistral/Mistral_7B_Instruct_v0.3/retrievals.csv
/content/drive/MyDrive/Bangla_Agriculture_FT_RAG_Mistral/Mistral_7B_Instruct_v0.3/retrievals.json
/content/drive/MyDrive/Bangla_Agriculture_FT_RAG_Mistral/Mistral_7B_Instruct_v0.3/predictions.csv
/content/drive/MyDrive/Bangla_Agriculture_FT_RAG_Mistral/Mistral_7B_Instruct_v0.3/result.csv
